# Wan 2.2 Генератор Видео — Colab T4

**Ноутбук «всё в одном»** для генерации видео с ComfyUI + Wan 2.2 14B на бесплатном T4 GPU.

| Воркфлоу | Вход | Выход |
|----------|------|-------|
| `video_wan_long_ultimate` | Изображение + текст | Длинное видео 20-30с (I2V + VACE) |
| `video_wan_t2v` | Только текст | Текст-в-видео |
| `video_wan_clip` | Изображение + текст | Короткий клип 3.4с для монтажа |
| `video_wan_v2v` | Видео + текст | Видео-в-видео рестайлинг |
| `video_wan_talking` | Фото + аудио | Говорящая голова (FantasyTalking) |
| `video_wan_reels` | Папки с фото | Reels из наборов изображений |

**Запускайте ячейки 1-6 по порядку**, затем откройте ссылку на туннель.

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Уже склонировано"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Кастомные ноды
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git 2>/dev/null || echo "Уже склонировано"
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git 2>/dev/null || echo "Уже склонировано"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Уже склонировано"

!pip install -r ComfyUI-WanVideoWrapper/requirements.txt -q
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

print("\nУстановлено!")

In [ ]:
#@title 3. Скачивание моделей (~28 ГБ, 5-10 мин)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)

HF = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"

files = {
    # Wan 2.2 T2V 14B fp8 (~14 ГБ)
    f"{M}/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors":
        f"{HF}/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
    # Модуль VACE 14B (~5.7 ГБ)
    f"{M}/diffusion_models/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors":
        f"{HF}/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
    # FantasyTalking (~0.5 ГБ)
    f"{M}/diffusion_models/fantasytalking_fp16.safetensors":
        f"{HF}/fantasytalking_fp16.safetensors",
    # Текстовый энкодер UMT5 XXL fp8 (~6.3 ГБ)
    f"{M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        f"{HF}/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    # VAE (~1.3 ГБ)
    f"{M}/vae/wan2.2_vae.safetensors":
        f"{HF}/wan2.2_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nВсе модели готовы!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/*

In [ ]:
#@title 4. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

workflows = [
    "video_wan_long_ultimate.json",
    "video_wan_t2v.json",
    "video_wan_clip.json",
    "video_wan_v2v.json",
    "video_wan_talking.json",
]

# Воркфлоу для Colab (фиксированные пути)
colab_workflows = {
    "video_wan_reels.json": "video_wan_reels_colab.json",
}

for wf in workflows:
    !wget -q -O "{WF_DIR}/{wf}" "{RAW}/{wf}"
    print(f"Скачано: {wf}")

for local_name, gist_name in colab_workflows.items():
    !wget -q -O "{WF_DIR}/{local_name}" "{RAW}/{gist_name}"
    print(f"Скачано: {gist_name} -> {local_name}")

print(f"\n{len(workflows) + len(colab_workflows)} воркфлоу сохранено в {WF_DIR}")

In [ ]:
#@title 5. Загрузка медиа (фото / аудио / видео)
#@markdown Загрузите файлы для ваших воркфлоу:
#@markdown - **Изображения** (.png, .jpg) для I2V, talking, clip воркфлоу
#@markdown - **Аудио** (.wav, .mp3) для talking воркфлоу
#@markdown - **Видео** (.mp4) для V2V воркфлоу

from google.colab import files
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(INPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Сохранено: {path}")

print(f"\nФайлы в input/: {os.listdir(INPUT_DIR)}")

In [ ]:
#@title 6. Запуск ComfyUI
import subprocess, time, re

# Установка cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Запуск ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("Запуск ComfyUI... (30с)")
time.sleep(30)

# Туннель Cloudflare
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI готов! Откройте: {url}")
    print(f"{'='*60}")
    print("\nЗагрузка воркфлоу: Меню -> Load -> выберите любой video_wan_* воркфлоу")
else:
    print("Ссылка на туннель не найдена. ComfyUI запущен на порту 8188.")

In [ ]:
#@title 7. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/wan"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("Результатов пока нет. Сначала сгенерируйте видео!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Скопировано: {os.path.basename(v)}")
    print(f"\n{len(videos)} файлов сохранено в {dst}")

---
## Гайд по воркфлоу

### `video_wan_long_ultimate` — Длинное I2V (20-30с)
Загрузите изображение -> задайте промпт -> Queue. Использует контекстные окна VACE для длинного связного видео.
- `num_frames`: 481=20с, 721=30с
- `blocks_to_swap`: 20 (по умолчанию для T4), увеличьте до 30 при OOM

### `video_wan_t2v` — Текст-в-видео
Только текстовый промпт, изображение не нужно. Чистая генерация по описанию.
- По умолчанию: 81 кадр (3.4с). Увеличьте для более длинных видео.

### `video_wan_clip` — Короткий клип (3.4с)
Быстрый клип из опционального стартового изображения. Идеален для генерации нескольких клипов под монтаж.

### `video_wan_v2v` — Видео-в-видео
Загрузите референсное видео -> рестайлинг текстовым промптом. VACE сохраняет движение/структуру.

### `video_wan_talking` — Говорящая голова
Загрузите портретное фото + аудиофайл -> генерирует видео говорящей головы, синхронизированное с речью.
- Используйте ячейку 8 ниже для генерации речи через TTS
- `audio_scale`: сила синхронизации губ (значение по умолчанию работает хорошо)

### `video_wan_reels` — Reels из папок
Загружает изображения из папок. Изначально для локального использования — на Colab загрузите изображения и настройте пути к папкам в воркфлоу.

### Пресеты разрешений
| Размер | Соотношение | Применение |
|--------|-------------|------------|
| 832x480 | 16:9 | Альбомная ориентация |
| 480x832 | 9:16 | Портрет/Reels |
| 608x1080 | 9:16 | HD Портрет |

---
## Дополнительные инструменты
Запускайте эти ячейки по необходимости — они не зависят от основной настройки.

In [ ]:
#@title 8. Генерация речи (TTS)
#@markdown Генерация аудио для воркфлоу **talking** с помощью edge-tts.

text = "Hello, this is a demo of the talking head generator." #@param {type:"string"}
voice = "en-US-AriaNeural" #@param ["en-US-AriaNeural", "en-US-GuyNeural", "ru-RU-SvetlanaNeural", "ru-RU-DmitryNeural"]
output_name = "speech.wav" #@param {type:"string"}

!pip install edge-tts -q 2>/dev/null
!edge-tts --voice "{voice}" --text "{text}" --write-media /content/ComfyUI/input/{output_name}

# Предпрослушивание
from IPython.display import Audio, display
display(Audio(f"/content/ComfyUI/input/{output_name}"))
print(f"\nСохранено: /content/ComfyUI/input/{output_name}")
print("Выберите этот файл в ноде LoadAudio в ComfyUI.")

In [ ]:
#@title 9. Улучшение промпта с Qwen2-VL
#@markdown Описание загруженного изображения с помощью Qwen2-VL 2B для генерации видео-промпта.
#@markdown Работает на CPU — занимает 1-2 мин.

image_name = "my_image.png" #@param {type:"string"}

!pip install transformers qwen-vl-utils -q 2>/dev/null

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print("Загрузка Qwen2-VL 2B на CPU...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="cpu"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

image_path = f"/content/ComfyUI/input/{image_name}"
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": f"file://{image_path}"},
        {"type": "text", "text": (
            "Describe this image in detail for use as a video generation prompt. "
            "Include: subject, action/pose, setting, lighting, colors, mood, "
            "camera angle. Write as one continuous paragraph, no bullet points."
        )}
    ]
}]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")

print("Генерация промпта...")
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256)

prompt = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)[0]

print(f"\n{'='*60}")
print("СГЕНЕРИРОВАННЫЙ ПРОМПТ (скопируйте в ComfyUI):")
print(f"{'='*60}")
print(prompt)

# Освобождение памяти
del model, processor
import gc; gc.collect()

In [ ]:
#@title 10. Монтаж: объединение видео
#@markdown Склеивание всех сгенерированных видео из output ComfyUI в один файл.

import glob, os

output_dir = "/content/ComfyUI/output"
videos = sorted(glob.glob(f"{output_dir}/*.mp4"))

if not videos:
    print("Видео не найдены в output/. Сначала сгенерируйте видео!")
else:
    print(f"Найдено {len(videos)} видео:")
    for i, v in enumerate(videos):
        print(f"  {i+1}. {os.path.basename(v)}")

    # Запись списка файлов для ffmpeg
    with open("/content/filelist.txt", "w") as f:
        for v in videos:
            f.write(f"file '{v}'\n")

    # Склеивание
    !ffmpeg -y -f concat -safe 0 -i /content/filelist.txt -c copy /content/montage.mp4 2>/dev/null

    print(f"\nМонтаж сохранён: /content/montage.mp4")
    !ls -lh /content/montage.mp4

    # Скачивание
    from google.colab import files
    files.download("/content/montage.mp4")

---
## Скоро: Пайплайн AI-музыкальных клипов

**Концепция:** Загрузите музыкальный трек и автоматически сгенерируйте синхронизированный музыкальный клип.

### Как это будет работать
1. **Детекция битов** (librosa) — анализ трека, поиск битов, сегментов, уровней энергии
2. **Генерация промптов** — создание видео-промпта для каждого сегмента на основе текста/настроения
3. **Генерация клипов** (ComfyUI API) — генерация короткого видеоклипа на каждый бит-сегмент через `video_wan_clip`
4. **Сборка** (FFmpeg) — склейка клипов, синхронизация с аудио, добавление переходов

### Ориентировочное время
- ~45-90 мин для 3-минутного видео на T4
- ~15-20 клипов по 3.4с каждый

### Стек
- `librosa` — детекция битов/onset, анализ темпа
- ComfyUI REST API — постановка в очередь video_wan_clip по сегментам
- `ffmpeg` — склейка + синхронизация аудио

**Это будет отдельный ноутбук: `colab_music_video.ipynb`**